# 🛡️ CyberBoyAI: URL Classifier Training (V5 - Production Parity)

**Key fix:** `extract_features()` here is **identical** to `backend/tools/url_tools.py`.
Previous versions had bugs (consec hardcoded to 0, wrong keyword lists).
This notebook is the definitive training source.

In [ ]:
!pip install scikit-learn==1.6.1 joblib tldextract httpx pandas

In [ ]:
import httpx, pandas as pd, zipfile, io, random

print('📥 Downloading malicious URLs...')
malicious_urls = []
try:
    with httpx.Client(follow_redirects=True, timeout=30) as client:
        malicious_urls.extend(client.get('https://openphish.com/feed.txt').text.strip().split('\n'))
        res = client.get('https://urlhaus.abuse.ch/downloads/csv_recent/')
        lines = [l for l in res.text.splitlines() if not l.startswith('#')]
        df = pd.read_csv(io.StringIO('\n'.join(lines)), names=['id','date','url','status','last','threat','tags','link','rep'])
        malicious_urls.extend(df['url'].tolist())
except Exception as e:
    print(f'Error: {e}')
malicious_urls = list(set([u for u in malicious_urls if u.startswith('http')]))
print(f'✅ {len(malicious_urls)} malicious URLs')

print('📥 Downloading benign URLs (Tranco)...')
benign_urls = []
try:
    res = httpx.get('https://tranco-list.eu/top-1m.csv.zip', timeout=60, follow_redirects=True)
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        with z.open('top-1m.csv') as f:
            df_b = pd.read_csv(f, names=['rank', 'domain'])
            subset = df_b['domain'].iloc[:len(malicious_urls)].tolist()
            paths = ['/', '/home', '/login', '/about', '/contact', '/api/v2', '/products']
            for d in subset:
                prefix = 'https://www.' if random.random() > 0.5 else 'https://'
                url = f'{prefix}{d}{random.choice(paths)}'
                benign_urls.append(url)
except Exception as e:
    print(f'Error: {e}')
print(f'✅ {len(benign_urls)} benign URLs (balanced)')

In [ ]:
# ============================================================
# PRODUCTION-IDENTICAL FEATURE EXTRACTION
# This is a copy of backend/tools/url_tools.py
# ALL changes to url_tools.py MUST be reflected here.
# ============================================================
import math, tldextract, re
from urllib.parse import urlparse

SHORTENERS = {'bit.ly','goo.gl','tinyurl.com','t.co','rebrand.ly','is.gd','buff.ly','ow.ly'}

def calculate_entropy(s):
    if not s: return 0.0
    prob = [float(s.count(c))/len(s) for c in dict.fromkeys(list(s))]
    return -sum([p * math.log(p)/math.log(2.0) for p in prob])

def get_vowel_consonant_ratio(s):
    v = sum(1 for c in s.lower() if c in 'aeiou')
    c = sum(1 for c in s.lower() if c.isalpha() and c not in 'aeiou')
    return v / c if c > 0 else 0.0

def get_max_consecutive_chars(s):
    if not s: return 0
    max_c, cur = 0, 1
    for i in range(1, len(s)):
        if s[i] == s[i-1]: cur += 1
        else: max_c = max(max_c, cur); cur = 1
    return max(max_c, cur)

def extract_features(url):
    """Extracts 17 features. MUST match backend/tools/url_tools.py exactly."""
    try:
        url = url.strip().rstrip('/')
        parsed = urlparse(url)
        ext = tldextract.extract(url)
        domain_part = ext.domain
        domain_full = f'{ext.domain}.{ext.suffix}'

        domain_age = 0  # Skipped at training for speed
        keywords = ['verify','secure','otp','login','update','bvn','bank','alert','signin','account']
        keyword_count = sum(1 for kw in keywords if kw in url.lower())
        entropy = calculate_entropy(domain_part)
        sub_clean = ext.subdomain.replace('www','').strip('.')
        subdomain_depth = len(sub_clean.split('.')) if sub_clean else 0
        is_https = 1 if parsed.scheme == 'https' else 0
        url_length = len(url)
        hyphen_count = domain_part.count('-')
        risky_tlds = ['.xyz','.tk','.top','.ml','.ga','.cf','.gq']
        tld_risk_score = 0.8 if any(ext.suffix.endswith(t[1:]) for t in risky_tlds) else 0.1
        special_char_count = sum(url.count(c) for c in ['@','=','?','_','&'])
        lookalikes = ['0','1','3','5']
        numeric_substitution = sum(domain_part.count(n) for n in lookalikes) / len(domain_part) if domain_part else 0.0
        percent_encoding_count = url.count('%')
        path_part = url[url.find('//')+2:]
        double_slash_redirect = 1 if '//' in path_part else 0
        is_ip_address = 1 if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', domain_part) else 0
        v_c_ratio = get_vowel_consonant_ratio(domain_part)
        consecutive_chars = get_max_consecutive_chars(domain_part)  # FIX: was hardcoded 0 in V4!
        is_shortened = 1 if domain_full in SHORTENERS else 0
        has_non_standard_port = 1 if parsed.port and parsed.port not in [80, 443] else 0

        # Return as ordered list matching ml_agent.py feature_names
        return [
            domain_age, keyword_count, entropy, subdomain_depth, is_https,
            url_length, hyphen_count, tld_risk_score, special_char_count, numeric_substitution,
            percent_encoding_count, double_slash_redirect, is_ip_address,
            v_c_ratio, consecutive_chars, is_shortened, has_non_standard_port
        ]
    except:
        return None

print('⚙️ Extracting features...')
data, labels = [], []
for u in malicious_urls:
    f = extract_features(u)
    if f: data.append(f); labels.append(1)
for u in benign_urls:
    f = extract_features(u)
    if f: data.append(f); labels.append(0)
print(f'✅ Dataset ready: {len(data)} samples ({labels.count(1)} phishing, {labels.count(0)} benign)')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

X, y = np.array(data), np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test)))

# Feature importance
names = ['domain_age','keyword_count','entropy','subdomain_depth','is_https',
         'url_length','hyphen_count','tld_risk_score','special_char_count','numeric_substitution',
         'percent_enc','double_slash','is_ip','v_c_ratio','consecutive_chars','is_shortened','non_std_port']
for n, imp in sorted(zip(names, clf.feature_importances_), key=lambda x: -x[1]):
    print(f'{n:25s}: {imp:.4f}')

# Sanity checks
print('\n--- SANITY CHECKS ---')
for t in ['https://www.google.com','https://gtbank.com','http://gtb4nk-verify.xyz','http://192.168.0.1/login','http://palmp4y-update.top/bvn']:
    f = extract_features(t)
    p = clf.predict_proba([f])[0][1]
    print(f'{p:.4f}  {t}')

In [ ]:
import joblib
joblib.dump(clf, 'model.joblib')
print('✅ Download model.joblib and place in backend/models/')